In [9]:
from torch.utils.data import Dataset
from pathlib import Path
import torch
import json
import re

### Fetching data

In [2]:
# Project location
project_dir = Path(r"C:\Users\samruddhi\Desktop\Sam\Gen_AI_Lab1")

# Dataset paths
data_dir = project_dir / "task1_llm" / "data"

train_file = data_dir / "tinystories_train_100k.txt"
val_file = data_dir / "tinystories_val_10k.txt"

# Output directory for processed data
processed_dir = project_dir / "task1_llm" / "Samruddhi" / "data_processed"
processed_dir.mkdir(parents=True, exist_ok=True)

In [3]:
def load_and_clean_stories(file_path):
    """
    Load one story per line and perform basic text cleaning.
    """
    stories = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            story = line.strip()

            # Skip completely empty lines
            if not story:
                continue

            # Replace repeated whitespace with one space
            story = re.sub(r"\s+", " ", story)

            stories.append(story)

    return stories

### Preprocessing the data

In [5]:
# Load training and validation stories
train_stories = load_and_clean_stories(train_file)
val_stories = load_and_clean_stories(val_file)

print("Data preprocessing completed.")
print(f"Training stories: {len(train_stories):,}")
print(f"Validation stories: {len(val_stories):,}")
print(f"First training story:\n{train_stories[0]}")

Data preprocessing completed.
Training stories: 99,983
Validation stories: 10,000
First training story:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt. Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt." Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


### Creating the character vocabulary

In [7]:
# Build the vocabulary using training data only
unique_characters = sorted(set("".join(train_stories)))

# Special token for characters that appear in validation
# but were not present in training
UNK_TOKEN = "<UNK>"

vocabulary = [UNK_TOKEN] + unique_characters

# Character-to-integer mapping
char_to_id = {
    character: index
    for index, character in enumerate(vocabulary)
}

# Integer-to-character mapping
id_to_char = {
    str(index): character
    for character, index in char_to_id.items()
}

# Save the vocabulary
vocab_file = processed_dir / "vocabulary.json"

with open(vocab_file, "w", encoding="utf-8") as file:
    json.dump(
        {
            "char_to_id": char_to_id,
            "id_to_char": id_to_char
        },
        file,
        indent=2,
        ensure_ascii=False
    )

print("Vocabulary created successfully.")
print(f"Number of unique characters: {len(unique_characters)}")
print(f"Vocabulary size including <UNK>: {len(vocabulary)}")
print(f"Vocabulary saved to: {vocab_file}")
print("\nSample mappings:")

for character in vocabulary[:20]:
    print(repr(character), "->", char_to_id[character])

Vocabulary created successfully.
Number of unique characters: 107
Vocabulary size including <UNK>: 108
Vocabulary saved to: C:\Users\samruddhi\Desktop\Sam\Gen_AI_Lab1\task1_llm\Samruddhi\data_processed\vocabulary.json

Sample mappings:
'<UNK>' -> 0
' ' -> 1
'!' -> 2
'"' -> 3
'#' -> 4
'$' -> 5
'&' -> 6
"'" -> 7
'(' -> 8
')' -> 9
'*' -> 10
'+' -> 11
',' -> 12
'-' -> 13
'.' -> 14
'/' -> 15
'0' -> 16
'1' -> 17
'2' -> 18
'3' -> 19


### Encode the text

In [8]:
def encode_text(text):
    """
    Convert characters into integer IDs.
    Unknown characters are mapped to <UNK>.
    """
    return [
        char_to_id.get(character, char_to_id[UNK_TOKEN])
        for character in text
    ]


# Join stories with a space between them
# so the model can distinguish where one story ends
train_text = " ".join(train_stories)
val_text = " ".join(val_stories)

# Convert text into integer IDs
train_token_ids = encode_text(train_text)
val_token_ids = encode_text(val_text)

# Store as compact PyTorch tensors
train_tokens = torch.tensor(train_token_ids, dtype=torch.long)
val_tokens = torch.tensor(val_token_ids, dtype=torch.long)

# Save encoded data
train_tokens_file = processed_dir / "train_tokens.pt"
val_tokens_file = processed_dir / "val_tokens.pt"

torch.save(train_tokens, train_tokens_file)
torch.save(val_tokens, val_tokens_file)

print("Text encoding completed.")
print(f"Training characters: {len(train_tokens):,}")
print(f"Validation characters: {len(val_tokens):,}")
print(f"First 20 training IDs: {train_tokens[:20].tolist()}")
print(f"Saved training tokens to: {train_tokens_file}")
print(f"Saved validation tokens to: {val_tokens_file}")

Text encoding completed.
Training characters: 87,906,298
Validation characters: 9,015,920
First 20 training IDs: [43, 71, 62, 1, 61, 58, 82, 12, 1, 58, 1, 69, 66, 77, 77, 69, 62, 1, 64, 66]
Saved training tokens to: C:\Users\samruddhi\Desktop\Sam\Gen_AI_Lab1\task1_llm\Samruddhi\data_processed\train_tokens.pt
Saved validation tokens to: C:\Users\samruddhi\Desktop\Sam\Gen_AI_Lab1\task1_llm\Samruddhi\data_processed\val_tokens.pt
